In [1]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor

from model import ViTYOLO

/Users/benosborn/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("rishitdagli/cppe-5")

ds, ds["train"][0]

(DatasetDict({
     train: Dataset({
         features: ['image_id', 'image', 'width', 'height', 'objects'],
         num_rows: 1000
     })
     test: Dataset({
         features: ['image_id', 'image', 'width', 'height', 'objects'],
         num_rows: 29
     })
 }),
 {'image_id': 15,
  'image': <PIL.Image.Image image mode=RGB size=943x663>,
  'width': 943,
  'height': 663,
  'objects': {'id': [114, 115, 116, 117],
   'area': [3796, 1596, 152768, 81002],
   'bbox': [[302.0, 109.0, 73.0, 52.0],
    [810.0, 100.0, 57.0, 28.0],
    [160.0, 31.0, 248.0, 616.0],
    [741.0, 68.0, 202.0, 401.0]],
   'category': [4, 4, 0, 0]}})

In [15]:
train, test = ds["train"], ds["test"]

features = train.features
categories = features["objects"].feature["category"].names

labels = {0: "None"}

for i, category in enumerate(categories):
    labels[i + 1] = category

num_labels = len(categories)

labels, num_labels

({0: 'None',
  1: 'Coverall',
  2: 'Face_Shield',
  3: 'Gloves',
  4: 'Goggles',
  5: 'Mask'},
 5)

In [4]:
checkpoint = "google/vit-base-patch16-224-in21k"

num_classes = num_labels
boxes_per_cell = 3
grid_size = 7
hidden_size = 1024

processor = AutoImageProcessor.from_pretrained(checkpoint)
model = ViTYOLO(num_classes, boxes_per_cell, grid_size, hidden_size) # add one class for the zero input

In [38]:
def preprocess_image(record, processor):
    processed = processor(images=record["image"], return_tensors="pt")

    return processed["pixel_values"]

def process_for_training(records, processor, num_classes, boxes_per_cell, grid_size):
    x_out = []
    y_out = []

    for record in records:
        try:
            # Calculate X
            tmp_x = preprocess_image(record, processor)

            # Calculate Y
            width = record["width"]
            height = record["height"]

            tmp_y = [[] for _ in range(grid_size * grid_size)]
            bbox = record["objects"]["bbox"]
            categories = record["objects"]["category"]

            cell_width = width // grid_size
            cell_height = height // grid_size

            for i, (x, y, dx, dy) in enumerate(bbox):
                center_x = (x + dx) / 2
                center_y = (y + dy) / 2

                # Calculate the result
                res = [0 for _ in range(4 + num_classes + 1)]
                res[0] = x / width
                res[1] = y / height
                res[2] = dx / width
                res[3] = dy / height

                res[4 + categories[i] + 1] = 1

                # Store the result
                cell_x = int(center_x / cell_width)
                cell_y = int(center_y / cell_height)

                idx = cell_y * grid_size + cell_x

                tmp_y[idx].append(res)

            # Ensure the number of boxes per cell are equal
            for i in range(len(tmp_y)):
                if len(tmp_y[i]) > boxes_per_cell:
                    tmp_y[i] = tmp_y[i][:boxes_per_cell]

                # Pad the results with empty boxes to meet box requirement
                diff = max(0, boxes_per_cell - len(tmp_y[i]))
                for _ in range(diff):
                    tmp_y[i].append([0 for _ in range(4 + num_classes + 1)])

            x_out.append(tmp_x)
            y_out.append(tmp_y)

        except Exception as e:
            print(f"exception {e} for record {record}, skipping")


    return x_out, y_out

out = process_for_training(ds["test"], processor, num_classes, boxes_per_cell, grid_size)

x, y = out
sample_x, sample_y, raw = x[0], y[0], ds["test"][0]

print(sample_x)

for grid in sample_y:
    print(grid)

print(raw)

exception Unable to infer channel dimension format for record {'image_id': 1027, 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=340x736 at 0x13FFFD4F0>, 'width': 340, 'height': 736, 'objects': {'id': [186, 187, 188, 189, 190, 191], 'area': [1222, 1764, 7104, 2668, 1836, 97000], 'bbox': [[152.0, 114.0, 47.0, 26.0], [147.0, 137.0, 42.0, 42.0], [133.0, 98.0, 74.0, 96.0], [82.0, 384.0, 46.0, 58.0], [218.0, 368.0, 34.0, 54.0], [61.0, 84.0, 194.0, 500.0]], 'category': [3, 4, 1, 2, 2, 0]}}, skipping
tensor([[[[0.2314, 0.2392, 0.2627,  ..., 0.2235, 0.2078, 0.1843],
          [0.2314, 0.2471, 0.2627,  ..., 0.2314, 0.2078, 0.1922],
          [0.2314, 0.2549, 0.2627,  ..., 0.2314, 0.2157, 0.2000],
          ...,
          [0.2157, 0.2314, 0.2549,  ..., 0.2627, 0.2627, 0.2392],
          [0.2235, 0.2314, 0.2471,  ..., 0.2627, 0.2471, 0.2392],
          [0.2157, 0.2235, 0.2392,  ..., 0.2627, 0.2471, 0.2392]],

         [[0.2392, 0.2471, 0.2706,  ..., 0.1922, 0.1843, 0.1686],
       

In [44]:
inputs = x[0]

with torch.no_grad():
    outputs = model(inputs)

inputs.shape, outputs, outputs.shape

(torch.Size([1, 3, 224, 224]),
 tensor([[[[-0.0490, -0.0559, -0.0181,  ..., -0.0684,  0.0496, -0.0171],
           [-0.0768,  0.0975,  0.0353,  ..., -0.0632,  0.0295, -0.0304],
           [-0.0236, -0.0611,  0.0158,  ..., -0.0157, -0.0197,  0.0046]],
 
          [[ 0.0745, -0.0610, -0.0066,  ..., -0.0134,  0.0257,  0.0509],
           [-0.0331,  0.0878, -0.0102,  ...,  0.0125,  0.0544,  0.0504],
           [ 0.0373, -0.1123,  0.0365,  ...,  0.0126,  0.0843,  0.0448]],
 
          [[-0.0475, -0.0139, -0.0247,  ...,  0.0317, -0.0495,  0.0217],
           [ 0.0257,  0.0298, -0.0053,  ..., -0.0533,  0.0179, -0.0465],
           [-0.0384,  0.0337,  0.0161,  ..., -0.0045,  0.0426, -0.0030]],
 
          ...,
 
          [[ 0.0568,  0.0060, -0.0152,  ..., -0.0021, -0.0304,  0.0305],
           [ 0.0329,  0.0192, -0.0071,  ...,  0.0347, -0.0048, -0.0925],
           [ 0.0089,  0.0423, -0.0055,  ...,  0.0720,  0.0197, -0.0234]],
 
          [[-0.0215,  0.0081,  0.0049,  ..., -0.0020, -0.0196, -